In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt


# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

In [ ]:


avonet = pd.read_csv( "../data/processed/avonet_cleaned.csv")


In [ ]:
avonet.columns

In [ ]:
df = avonet.copy()

# Dimorphism

In [ ]:
# features that have _avg_m and _avg_f splits
dimorphism_features = {
    'beak_culmen' : ('beak_culmen_avg_m', 'beak_culmen_avg_f', 'beak_culmen_avg'),
    'beak_nares'  : ('beak_nares_avg_m',  'beak_nares_avg_f',  'beak_nares_avg'),
    'beak_width'  : ('beak_width_avg_m',  'beak_width_avg_f',  'beak_width_avg'),
    'beak_depth'  : ('beak_depth_avg_m',  'beak_depth_avg_f',  'beak_depth_avg'),
    'tarsus'      : ('tarsus_avg_m',      'tarsus_avg_f',      'tarsus_avg'),
    'wing_len'    : ('wing_len_avg_m',    'wing_len_avg_f',    'wing_len_avg'),
    'kipps'       : ('kipps_avg_m',       'kipps_avg_f',       'kipps_avg'),
    'secondary'   : ('secondary_avg_m',   'secondary_avg_f',   'secondary_avg'),
    'hwi'         : ('hwi_avg_m',         'hwi_avg_f',         'hwi_avg'),
    'tail'        : ('tail_avg_m',        'tail_avg_f',        'tail_avg'),
}

for feature, (male_col, female_col, avg_col) in dimorphism_features.items():
    null_mask = (
        df[male_col].isnull()   |
        df[female_col].isnull() |
        df[avg_col].isnull()    |
        (df[avg_col] == 0)
    )
    df[f'dimorphism_{feature}'] = (df[male_col] - df[female_col]) / df[avg_col]
    df.loc[null_mask, f'dimorphism_{feature}'] = np.nan

In [ ]:
m_f_cols = [col for col in df.columns if col.endswith('_avg_m') or col.endswith('_avg_f')]
df.drop(columns=m_f_cols, inplace=True)

# Log Tranform

In [ ]:
df['log_mass'] = np.where(
    df['mass_avg'].isnull() | (df['mass_avg'] <= 0),
    np.nan,
    np.log(df['mass_avg'])
)

In [ ]:
df['log_RangeSize'] = np.where(
    df['range_size'].isnull() | (df['range_size'] <= 0),
    np.nan,
    np.log(df['range_size'])
)

In [ ]:
print(df[['mass_avg', 'log_mass']].describe())

In [ ]:
print(df[['range_size', 'log_RangeSize']].describe())

# Wing pointedness

In [ ]:
# Wing pointedness = kipps / wing_length

In [ ]:
df['wing_pointedness'] = np.where(
    df['wing_len_avg'].isnull() | df['kipps_avg'].isnull() | (df['wing_len_avg'] == 0),
    np.nan,
    df['kipps_avg'] / df['wing_len_avg']
)

In [ ]:
print(df[['wing_len_avg', 'kipps_avg', 'wing_pointedness']].describe())

# Wing loading index 

In [ ]:
# Wing loading index = mass / wing_length²

In [ ]:
df['wing_loading'] = np.where(
    df['mass_avg'].isnull() | df['wing_len_avg'].isnull() | (df['wing_len_avg'] == 0),
    np.nan,
    df['mass_avg'] / (df['wing_len_avg'] ** 2)
)

In [ ]:
print(df[['mass_avg', 'wing_len_avg', 'wing_loading']].describe())

In [ ]:
df['log_wing_loading'] = np.where(
    df['wing_loading'].isnull() | (df['wing_loading'] <= 0),
    np.nan,
    np.log(df['wing_loading'])
)

print(df[['wing_loading', 'log_wing_loading']].describe())

# Beak Shape 

In [ ]:
# Beak Shape : Beak elongation ratio = culmen / width

In [ ]:
df['beak_elongation'] = np.where(
    df['beak_culmen_avg'].isnull() | df['beak_width_avg'].isnull() | (df['beak_width_avg'] == 0),
    np.nan,
    df['beak_culmen_avg'] / df['beak_width_avg']
)

print(df[['beak_culmen_avg', 'beak_width_avg', 'beak_elongation']].describe())

# Beak morphology 

In [ ]:
# Beak morphology : Beak robustness index = depth × width / culmen  ;

In [ ]:
df['beak_robustness'] = np.where(
    df['beak_depth_avg'].isnull() | df['beak_width_avg'].isnull() | df['beak_culmen_avg'].isnull() | (df['beak_culmen_avg'] == 0),
    np.nan,
    (df['beak_depth_avg'] * df['beak_width_avg']) / df['beak_culmen_avg']
)

print(df[['beak_depth_avg', 'beak_width_avg', 'beak_culmen_avg', 'beak_robustness']].describe())

# Absolute latitude

In [ ]:
# Absolute latitude = |latitude| 

In [ ]:
df['abs_lat_min']      = df['lat_min'].abs()
df['abs_lat_max']      = df['lat_max'].abs()
df['abs_lat_centroid'] = df['lat_centroid'].abs()

print(df[['lat_min', 'abs_lat_min',
          'lat_max', 'abs_lat_max',
          'lat_centroid', 'abs_lat_centroid']].describe())

# Aspect ratio 

In [ ]:
# Aspect ratio =  wing_length / secondary

In [ ]:
df['aspect_ratio'] = np.where(
    df['wing_len_avg'].isnull() | df['secondary_avg'].isnull() | (df['secondary_avg'] == 0),
    np.nan,
    df['wing_len_avg'] / df['secondary_avg']
)

print(df[['wing_len_avg', 'secondary_avg', 'aspect_ratio']].describe())

# Body condition

In [ ]:
# Body condition = mass / tarsus³

In [ ]:
df['body_condition'] = np.where(
    df['mass_avg'].isnull() | df['tarsus_avg'].isnull() | (df['tarsus_avg'] == 0),
    np.nan,
    df['mass_avg'] / (df['tarsus_avg'] ** 3)
)

print(df[['mass_avg', 'tarsus_avg', 'body_condition']].describe())

# Tail-to-wing ratio

In [ ]:
# Tail-to-wing ratio = tail_length / wing_length

In [ ]:
df['tail_to_wing'] = np.where(
    df['tail_avg'].isnull() | df['wing_len_avg'].isnull() | (df['wing_len_avg'] == 0),
    np.nan,
    df['tail_avg'] / df['wing_len_avg']
)

print(df[['tail_avg', 'wing_len_avg', 'tail_to_wing']].describe())

# Climate zone

In [ ]:
# Climate zone bin |lat| → tropical /temperate / polar

In [ ]:
def classify_climate_zone(lat):
    if pd.isnull(lat):
        return np.nan
    elif lat <= 23.5:
        return 'tropical'
    elif lat <= 60:
        return 'temperate'
    else:
        return 'polar'

df['climate_zone'] = df['abs_lat_centroid'].apply(classify_climate_zone)

print(df['climate_zone'].value_counts())

In [ ]:
df.columns

In [ ]:
df.rename(columns={
    # beak
    'beak_culmen_avg' : 'beak_culmen',
    'beak_nares_avg'  : 'beak_nares',
    'beak_width_avg'  : 'beak_width',
    'beak_depth_avg'  : 'beak_depth',

    # tarsus
    'tarsus_avg'      : 'tarsus',

    # wing
    'wing_len_avg'    : 'wing_len',
    'kipps_avg'       : 'kipps',
    'secondary_avg'   : 'secondary',
    'hwi_avg'         : 'hwi',

    # tail
    'tail_avg'        : 'tail',

    # mass
    'mass_avg'        : 'mass',
}, inplace=True)


# reorder columns — native feature followed by its engineered features
ordered_cols = [
    # ids / taxonomy
    'avibase_id', 'species_birdlife','species_birdtree',
    'family_birdlife', 'order_birdlife',
    'family_birdtree', 'order_birdtree',

    # beak
    'beak_culmen', 'beak_nares', 'beak_width', 'beak_depth',
    'dimorphism_beak_culmen', 'dimorphism_beak_nares',
    'dimorphism_beak_width', 'dimorphism_beak_depth',
    'beak_elongation', 'beak_robustness',

    # tarsus
    'tarsus', 'dimorphism_tarsus',

    # wing
    'wing_len', 'kipps', 'secondary', 'hwi',
    'dimorphism_wing_len', 'dimorphism_kipps',
    'dimorphism_secondary', 'dimorphism_hwi',
    'wing_pointedness', 'wing_loading', 'log_wing_loading',
    'aspect_ratio',

    # tail
    'tail', 'dimorphism_tail', 'tail_to_wing',

    # mass
    'mass', 'log_mass', 'body_condition',

    # counts
    'total_individuals', 'female_count', 'male_count',
    'mass_source', 'inference',

    # geographic
    'lat_min', 'abs_lat_min',
    'lat_max', 'abs_lat_max',
    'lat_centroid', 'abs_lat_centroid',
    'lon_centroid', 'range_size', 'log_RangeSize',
    'climate_zone',

    # categorical
    'habitat', 'habitat_density', 'migration',
    'trophic_level', 'trophic_niche', 'lifestyle',
]

df = df[ordered_cols]


In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.drop(columns=["dimorphism_beak_depth"],inplace=True)

In [ ]:
mapping = {
    1: "Open",
    2: "Semi-dense",
    3: "Dense"
}

df["habitat_density"] = df["habitat_density"].map(mapping)

In [ ]:
df["habitat_density"]

In [ ]:
migration_map = {1: "resident", 2: "partial", 3: "migratory"}
df["migration"] = df["migration"].map(migration_map)

In [ ]:
df.to_csv('../data/processed/avonet_FE_02.csv', index=False , float_format='%.5f')

# EDA

In [ ]:
morph_features = [
    'beak_culmen', 'beak_nares', 'beak_width', 'beak_depth',
    'beak_elongation', 'beak_robustness', 'tarsus',
    'wing_len', 'kipps', 'secondary', 'hwi',
    'wing_pointedness', 'wing_loading', 'log_wing_loading',
    'aspect_ratio', 'tail', 'tail_to_wing',
    'log_mass', 'body_condition'
]

In [ ]:
corr_matrix = df[morph_features].corr(method='spearman')

plt.figure(figsize=(16, 13))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7}
)
plt.title("Spearman Correlation - Morphological Features", fontsize=14)
plt.tight_layout()
plt.savefig("step1_morph_correlation_heatmap.png", dpi=150)
plt.show()

## conclusion1

* Cluster 1 — Beak Size (beak_culmen, beak_nares, beak_width, beak_depth, beak_robustness)
All strongly correlated 0.65–0.94. These are not independent features, they all measure the same thing — overall beak size. One big beak means everything is big together.
The exception — beak_elongation
Negatively correlated with beak_width (-0.47), beak_depth (-0.34), and strongly negative with beak_robustness (-0.70). This is biologically meaningful — slender elongated beaks are fundamentally a different shape strategy, not just smaller versions of robust beaks.

* Cluster 2 — Wing Shape (hwi, wing_pointedness, aspect_ratio)
Nearly perfectly correlated — 0.98 to 1.00. These three are essentially the same measurement. Kipps also joins this group at 0.88. This means in your analysis you only need one of these to represent wing pointedness.


* Cluster 3 — Body Size (log_mass, wing_len, secondary, tarsus)
All highly correlated 0.78–0.94. Bigger birds have longer wings and longer legs — classic allometric scaling.
Interesting independence
beak_elongation is almost completely independent of wing traits and mass (near zero correlations). This means beak shape strategy evolved separately from body size and flight style — very interesting for your ecological analysis.
tail_to_wing
Negatively correlated with all wing shape metrics (-0.50 to -0.52). Birds with pointed wings have proportionally shorter tails.


---
These clusters tells  that , when you go feature → ecology,
we should use one representative per cluster rather than all correlated features. That avoids redundancy in conclusions.

## Step 2: Beak Morphology vs Trophic Niche

### Objective
Test the core form-function hypothesis: do birds with different dietary strategies 
evolve distinct beak shapes?

### Features Selected (based on Step 1 findings)
From the correlation heatmap we identified two independent beak axes:
- **beak_culmen** → represents overall beak length (size axis)
- **beak_depth** → represents beak bulk (size axis, highly correlated cluster)
- **beak_elongation** → represents slenderness (shape axis, independent from size)
- **beak_robustness** → represents beak stoutness (negatively correlated with elongation)

### Why these four?
beak_width is redundant with beak_depth (r=0.92). beak_nares is redundant with 
beak_culmen (r=0.94). We drop redundant features and keep one per dimension.

### Method
Boxplot of each beak feature grouped by `trophic_niche`.  
Outliers hidden for visual clarity — we are comparing medians and spread across groups.

In [ ]:
# Two most meaningful beak representatives from Step 1
beak_features = ['beak_culmen', 'beak_depth', 'beak_elongation', 'beak_robustness']

# Drop rows where trophic_niche is missing
df_trophic = df[df['trophic_niche'].notna()].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(beak_features):
    order = df_trophic.groupby('trophic_niche')[feat].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_trophic,
        x='trophic_niche',
        y=feat,
        order=order,
        ax=axes[i],
        palette='Set2',
        showfliers=False
    )
    axes[i].set_title(f'{feat} by Trophic Niche', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle("Beak Morphology across Trophic Niches", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("step2_beak_trophic.png", dpi=150)
plt.show()

### Observations

- **beak_culmen:** Scavengers > Aquatic predators >> all others. Predatory feeding 
  strategies require physically longer beaks. Granivores and Nectarivores have 
  the shortest beaks.

- **beak_depth:** Scavengers have disproportionately deep beaks (~32mm median), 
  nearly double Vertivores. Nectarivores approach zero depth — extreme opposite end.

- **beak_elongation:** Nectarivores and Aquatic predators lead. Both need long slender 
  beaks for very different reasons — flower probing vs fish spearing. Granivores 
  and terrestrial herbivores have the most compact beak shapes.

- **beak_robustness:** Perfectly mirrors elongation in reverse. Scavengers most robust, 
  Nectarivores near-zero. Confirms the -0.70 anti-correlation found in Step 1.

### Conclusion
Trophic niche is a strong predictor of beak morphology. Two distinct beak 
strategies emerge clearly:
1. **Size + Robustness strategy** — Scavengers, Vertivores: large, deep, stout beaks 
   for force-based feeding
2. **Elongation strategy** — Nectarivores, Aquatic predators: long, slender beaks 
   for precision-based feeding

These two strategies sit at opposite ends of the elongation-robustness axis 
identified in Step 1 (r = -0.70), confirming that beak shape is functionally 
constrained by diet.

## Step 3: Wing Morphology vs Lifestyle and Migration

### Objective
Test whether birds with active aerial lifestyles or migratory behaviour have 
evolved more efficient wing shapes — more pointed, higher aspect ratio, 
higher wing loading.

### Features Selected (based on Step 1 findings)
From Step 1 we found hwi, wing_pointedness, aspect_ratio are nearly identical (r=0.98-1.00).
So we use:
- **hwi** → represents wing pointedness (one representative for the whole cluster)
- **wing_loading** → represents flight efficiency / power requirement
- **tail_to_wing** → represents maneuverability trade-off

### Grouping Variables
- **lifestyle** → aerial, ground-living, aquatic etc
- **migration** → migratory vs resident

### Method
Boxplots for lifestyle grouping.
Side by side boxplots for migration comparison.
Outliers hidden for visual clarity.

In [ ]:
wing_features = ['hwi', 'wing_loading', 'tail_to_wing']

# ── Plot 1: Wing traits vs Lifestyle ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, feat in enumerate(wing_features):
    df_life = df[['lifestyle', feat]].dropna()
    order = df_life.groupby('lifestyle')[feat].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_life,
        x='lifestyle',
        y=feat,
        order=order,
        ax=axes[i],
        palette='Set1',
        showfliers=False
    )
    axes[i].set_title(f'{feat} by Lifestyle', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle("Wing Morphology across Lifestyles", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("step3a_wing_lifestyle.png", dpi=150)
plt.show()

# ── Plot 2: Wing traits vs Migration ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, feat in enumerate(wing_features):
    df_mig = df[['migration', feat]].dropna()
    order = df_mig.groupby('migration')[feat].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_mig,
        x='migration',
        y=feat,
        order=order,
        ax=axes[i],
        palette='Set2',
        showfliers=False
    )
    axes[i].set_title(f'{feat} by Migration', fontsize=12)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle("Wing Morphology across Migration Strategies", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("step3b_wing_migration.png", dpi=150)
plt.show()

### Observations

**Lifestyle:**
- **hwi:** Aerial birds have highest wing pointedness (median ~62), 
  confirming aerially adapted species evolved more efficient wing shapes. 
  Insessorial birds lowest.
- **wing_loading:** Aquatic birds highest (~0.02) — heavy bodies require 
  more power. Aerial birds lowest — light and efficient fliers.
- **tail_to_wing:** Aquatic birds have shortest relative tails. 
  Generalists longest — need maneuverability over speed.


### Conclusion
Lifestyle is a strong predictor of wing morphology. Aerial birds confirm 
the pointed-wing hypothesis via hwi, but wing_loading reveals a separate 
dimension — body mass driven flight cost — where aquatic birds dominate. 
Two distinct wing strategies: efficiency-optimised (Aerial) vs 
power-optimised (Aquatic).

## Step 4: Body Size vs Latitude (Bergmann's Rule)

### Objective
Test Bergmann's Rule — a classic ecological pattern where birds living at 
higher latitudes (closer to poles) tend to have larger body sizes, as larger 
bodies retain heat more efficiently in colder climates.

### Features Selected
- **log_mass** → body size representative (log transformed for normality)
- **abs_lat_centroid** → absolute latitude of species range centroid 
  (0 = equator, 90 = pole, sign removed so both hemispheres treated equally)
- **abs_lat_min, abs_lat_max** → range extent for additional context
- **climate_zone** → categorical grouping for supplementary view

### Method
- Scatter plot of log_mass vs abs_lat_centroid with regression line
- Boxplot of log_mass grouped by climate_zone
- Null values dropped per column before plotting

In [ ]:
from scipy import stats

# ── Plot 1: Scatter log_mass vs abs_lat_centroid with regression ───────────────
df_lat = df[['log_mass', 'abs_lat_centroid']].dropna()

slope, intercept, r_value, p_value, std_err = stats.linregress(
    df_lat['abs_lat_centroid'], df_lat['log_mass']
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Scatter + regression line
axes[0].scatter(
    df_lat['abs_lat_centroid'], df_lat['log_mass'],
    alpha=0.3, s=10, color='steelblue'
)
x_line = np.linspace(df_lat['abs_lat_centroid'].min(), df_lat['abs_lat_centroid'].max(), 100)
axes[0].plot(x_line, slope * x_line + intercept, color='red', linewidth=2)
axes[0].set_xlabel('Absolute Latitude (centroid)', fontsize=12)
axes[0].set_ylabel('log_mass', fontsize=12)
axes[0].set_title(
    f'Body Size vs Latitude\nr = {r_value:.3f}, p = {p_value:.4f}',
    fontsize=12
)

# ── Plot 2: log_mass by climate_zone ──────────────────────────────────────────
df_clim = df[['log_mass', 'climate_zone']].dropna()
order = df_clim.groupby('climate_zone')['log_mass'].median().sort_values(ascending=False).index

sns.boxplot(
    data=df_clim,
    x='climate_zone',
    y='log_mass',
    order=order,
    ax=axes[1],
    palette='coolwarm',
    showfliers=False
)
axes[1].set_title('Body Size across Climate Zones', fontsize=12)
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle("Bergmann's Rule — Body Size vs Latitude and Climate", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("step4_bergmann.png", dpi=150)
plt.show()

print(f"\nRegression Summary:")
print(f"  Slope     : {slope:.4f}")
print(f"  R value   : {r_value:.4f}")
print(f"  R-squared : {r_value**2:.4f}")
print(f"  P value   : {p_value:.6f}")

### Observations

- **Scatter (log_mass vs abs_lat_centroid):** Positive slope (0.0161) confirms 
  Bergmann's rule direction — body size increases with latitude. However R = 0.152 
  and R-squared = 0.023 indicate latitude explains only 2.3% of body size variance. 
  Massive scatter at low latitudes driven by tropical diversity.

- **R value:** 0.152 — weak but consistent positive correlation
- **P value:** 0.000000 — statistically significant due to large sample size, 
  not due to effect strength. Effect size is small.

- **Climate zone boxplot:** Polar birds median log_mass ~5 (≈150g), clearly 
  heavier than temperate and tropical birds (median ~3.8, ≈45g). 
  Temperate and tropical zones nearly identical in median.

### Conclusion
Bergmann's Rule is statistically confirmed but ecologically weak in this dataset. 
Latitude alone is a poor predictor of body size (R² = 0.023). The polar vs 
non-polar contrast in the climate zone plot is the clearest signal — polar birds 
are systematically heavier, but within temperate and tropical zones body size 
is driven by other factors such as diet, lifestyle, and taxonomy rather than 
geography alone. Large sample size inflates statistical significance here — 
this is an important data interpretation caveat for the brief's uncertainty 
requirement.

## PCA: Morphological Space Exploration

### Objective
Reduce correlated morphological features into 2D space to visually identify:
1. How beak shape clusters by trophic niche
2. How overall morphology clusters by taxonomic order

### Why PCA here
From Step 1 we found many features are highly correlated — beak features 
cluster together, wing features cluster together. PCA collapses these 
correlated groups into independent axes so we can plot all species in 2D 
and see natural groupings.

### Two PCA runs
- **PCA 1:** Beak features only → coloured by trophic_niche
- **PCA 2:** All morphological features → coloured by order_birdlife (top 8 orders)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ── PCA 1: Beak features coloured by trophic_niche ───────────────────────────
beak_cols = ['beak_culmen', 'beak_nares', 'beak_width', 
             'beak_depth', 'beak_elongation', 'beak_robustness']

df_beak = df[beak_cols + ['trophic_niche']].dropna()

scaler = StandardScaler()
X_beak = scaler.fit_transform(df_beak[beak_cols])

pca_beak = PCA(n_components=2)
components_beak = pca_beak.fit_transform(X_beak)

df_beak['PC1'] = components_beak[:, 0]
df_beak['PC2'] = components_beak[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

niches = df_beak['trophic_niche'].unique()
palette = plt.cm.Set2(np.linspace(0, 1, len(niches)))

for j, niche in enumerate(niches):
    mask = df_beak['trophic_niche'] == niche
    axes[0].scatter(
        df_beak.loc[mask, 'PC1'],
        df_beak.loc[mask, 'PC2'],
        label=niche, alpha=0.4, s=10, color=palette[j]
    )

axes[0].set_xlabel(f"PC1 ({pca_beak.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
axes[0].set_ylabel(f"PC2 ({pca_beak.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
axes[0].set_title("Beak Morphology PCA — coloured by Trophic Niche", fontsize=12)
axes[0].legend(markerscale=3, fontsize=8, loc='upper right')

# ── PCA 2: All morphological features coloured by order ───────────────────────
morph_cols = [
    'beak_culmen', 'beak_nares', 'beak_width', 'beak_depth',
    'beak_elongation', 'beak_robustness', 'tarsus',
    'wing_len', 'kipps', 'secondary', 'hwi',
    'wing_pointedness', 'wing_loading', 'aspect_ratio',
    'tail', 'tail_to_wing', 'log_mass'
]

top8_orders = df['order_birdlife'].value_counts().head(8).index
df_morph = df[morph_cols + ['order_birdlife']].dropna()
df_morph = df_morph[df_morph['order_birdlife'].isin(top8_orders)]

X_morph = scaler.fit_transform(df_morph[morph_cols])

pca_morph = PCA(n_components=2)
components_morph = pca_morph.fit_transform(X_morph)

df_morph['PC1'] = components_morph[:, 0]
df_morph['PC2'] = components_morph[:, 1]

orders = df_morph['order_birdlife'].unique()
palette2 = plt.cm.tab10(np.linspace(0, 1, len(orders)))

for j, order in enumerate(orders):
    mask = df_morph['order_birdlife'] == order
    axes[1].scatter(
        df_morph.loc[mask, 'PC1'],
        df_morph.loc[mask, 'PC2'],
        label=order, alpha=0.4, s=10, color=palette2[j]
    )

axes[1].set_xlabel(f"PC1 ({pca_morph.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
axes[1].set_ylabel(f"PC2 ({pca_morph.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
axes[1].set_title("Full Morphology PCA — coloured by Taxonomic Order", fontsize=12)
axes[1].legend(markerscale=3, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig("pca_analysis.png", dpi=150)
plt.show()

# ── Print variance explained ──────────────────────────────────────────────────
print("=== PCA 1 — Beak Features ===")
print(f"PC1 variance explained: {pca_beak.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 variance explained: {pca_beak.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total (2 components)  : {sum(pca_beak.explained_variance_ratio_)*100:.1f}%")

print("\n=== PCA 2 — Full Morphology ===")
print(f"PC1 variance explained: {pca_morph.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 variance explained: {pca_morph.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total (2 components)  : {sum(pca_morph.explained_variance_ratio_)*100:.1f}%")

In [ ]:
# drop : dimorphism_beak_depth , mass ? ,